# Model Training: LOOCV + Optuna


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import joblib
from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold, cross_val_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from tqdm import tqdm


In [2]:
DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

TARGETS = {
    'ODI':      'odi_post',
    'SF36_PCS': 'sf36_phys_post',
    'SF36_MCS': 'sf36_psych_post',
}
MCID = {'ODI': 10, 'SF36_PCS': 5, 'SF36_MCS': 5}

CAT_COLS      = ['sex', 'neuro_deficit', 'facet_tropism']
MODEL_NAMES   = ['Ridge', 'Lasso', 'ElasticNet', 'XGBoost', 'LightGBM', 'CatBoost']
N_TRIALS      = 30
N_TRIALS_GB   = 20
N_INNER_FOLDS = 5
RANDOM_STATE  = 42


In [3]:
X_raw = pd.read_csv(DATA_DIR / 'X.csv')
y_raw = pd.read_csv(DATA_DIR / 'y.csv')
n     = len(X_raw)

NUM_COLS = [c for c in X_raw.columns if c not in CAT_COLS]

def make_preprocessor():
    return ColumnTransformer([
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OneHotEncoder(drop='first', sparse_output=False,
                              handle_unknown='ignore'), CAT_COLS),
    ])

_prep = make_preprocessor()
_prep.fit(X_raw)
FEATURE_NAMES = (
    NUM_COLS
    + list(_prep.named_transformers_['cat'].get_feature_names_out(CAT_COLS))
)
print(f'Patients: {n}, Features: {X_raw.shape[1]} → after OHE: {len(FEATURE_NAMES)}')


Patients: 500, Features: 27 → after OHE: 27


In [4]:
def make_objective(model_name, X_tr, y_tr):
    kf = KFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        if model_name == 'Ridge':
            model = Ridge(
                alpha=trial.suggest_float('alpha', 1e-3, 1e3, log=True)
            )
        elif model_name == 'Lasso':
            model = Lasso(
                alpha=trial.suggest_float('alpha', 1e-4, 100, log=True),
                max_iter=10_000,
            )
        elif model_name == 'ElasticNet':
            model = ElasticNet(
                alpha=trial.suggest_float('alpha', 1e-4, 100, log=True),
                l1_ratio=trial.suggest_float('l1_ratio', 0.05, 0.95),
                max_iter=10_000,
            )
        elif model_name == 'XGBoost':
            model = XGBRegressor(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                max_depth=trial.suggest_int('max_depth', 3, 8),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                subsample=trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
                reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 2.0, log=True),
                reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 4.0, log=True),
                random_state=RANDOM_STATE, verbosity=0,
                nthread=1, tree_method='hist',
            )
        elif model_name == 'LightGBM':
            model = LGBMRegressor(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                max_depth=trial.suggest_int('max_depth', 3, 8),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                num_leaves=trial.suggest_int('num_leaves', 20, 80),
                subsample=trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
                reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 2.0, log=True),
                reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 4.0, log=True),
                random_state=RANDOM_STATE, verbose=-1, bagging_freq=1,
                n_jobs=1,
            )
        elif model_name == 'CatBoost':
            model = CatBoostRegressor(
                iterations=trial.suggest_int('iterations', 50, 200),
                depth=trial.suggest_int('depth', 3, 8),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
                random_seed=RANDOM_STATE, verbose=0,
                thread_count=1, border_count=64,
            )

        scores = cross_val_score(
            model, X_tr, y_tr,
            cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return -scores.mean()

    return objective


def fit_best_model(model_name, params, X_tr, y_tr):
    if model_name == 'Ridge':
        model = Ridge(**params)
    elif model_name == 'Lasso':
        model = Lasso(**params, max_iter=10_000)
    elif model_name == 'ElasticNet':
        model = ElasticNet(**params, max_iter=10_000)
    elif model_name == 'XGBoost':
        model = XGBRegressor(**params, random_state=RANDOM_STATE, verbosity=0,
                             nthread=1, tree_method='hist')
    elif model_name == 'LightGBM':
        model = LGBMRegressor(**params, random_state=RANDOM_STATE, verbose=-1, bagging_freq=1,
                              n_jobs=1)
    elif model_name == 'CatBoost':
        model = CatBoostRegressor(**params, random_seed=RANDOM_STATE, verbose=0,
                                  thread_count=1, border_count=64)
    model.fit(X_tr, y_tr)
    return model


In [5]:
checkpoint_file = RESULTS_DIR / 'loocv_checkpoint.pkl'
if checkpoint_file.exists():
    with open(checkpoint_file, 'rb') as f:
        checkpoint = pickle.load(f)
else:
    checkpoint = {}

indices = np.arange(n)

for target_key, target_col in TARGETS.items():
    y_all = y_raw[target_col].values.astype(float)

    for model_name in MODEL_NAMES:
        key = (target_key, model_name)
        if key in checkpoint:
            continue

        y_pred = np.zeros(n)

        for i in tqdm(range(n), desc=f'{target_key} / {model_name}', leave=True):
            mask_train = indices != i

            X_tr_raw = X_raw.iloc[mask_train].reset_index(drop=True)
            X_te_raw = X_raw.iloc[[i]].reset_index(drop=True)
            y_tr     = y_all[mask_train]

            prep = make_preprocessor()
            X_tr_sc = prep.fit_transform(X_tr_raw)
            X_te_sc = prep.transform(X_te_raw)

            study = optuna.create_study(
                direction='minimize',
                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
            )
            n_trials = N_TRIALS_GB if model_name in ('XGBoost', 'LightGBM', 'CatBoost') else N_TRIALS
            study.optimize(
                make_objective(model_name, X_tr_sc, y_tr),
                n_trials=n_trials,
                show_progress_bar=False,
            )

            model = fit_best_model(model_name, study.best_params, X_tr_sc, y_tr)
            y_pred[i] = model.predict(X_te_sc)[0]

        checkpoint[key] = y_pred
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint, f)

        mae = np.mean(np.abs(y_pred - y_all))
        print(f'  Saved {target_key}/{model_name}  MAE = {mae:.3f}')

loocv_predictions = {}
for target_key, target_col in TARGETS.items():
    y_all = y_raw[target_col].values.astype(float)
    loocv_predictions[target_key] = {
        'y_true': y_all,
        'y_pred': {
            model_name: checkpoint[(target_key, model_name)]
            for model_name in MODEL_NAMES
        },
    }


ODI / Ridge: 100%|██████████| 500/500 [03:27<00:00,  2.41it/s]


  Saved ODI/Ridge  MAE = 5.595


ODI / Lasso: 100%|██████████| 500/500 [04:00<00:00,  2.08it/s]


  Saved ODI/Lasso  MAE = 5.573


ODI / ElasticNet: 100%|██████████| 500/500 [05:25<00:00,  1.54it/s]


  Saved ODI/ElasticNet  MAE = 5.579


... (26 outputs left out)

SF36_MCS / LightGBM: 100%|██████████| 500/500 [05:35<00:00,  1.49it/s]


  Saved SF36_MCS/LightGBM  MAE = 5.842


SF36_MCS / CatBoost: 100%|██████████| 500/500 [50:10<00:00,  6.02s/it]

  Saved SF36_MCS/CatBoost  MAE = 5.511


In [6]:
final_preprocessor = make_preprocessor()
X_all_sc = final_preprocessor.fit_transform(X_raw)

final_models = {}
final_params = {}

for target_key, target_col in TARGETS.items():
    y_all = y_raw[target_col].values.astype(float)
    final_models[target_key] = {}
    final_params[target_key] = {}

    for model_name in tqdm(MODEL_NAMES, desc=f'Final models: {target_key}'):
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        )
        n_trials = N_TRIALS_GB if model_name in ('XGBoost', 'LightGBM', 'CatBoost') else N_TRIALS
        study.optimize(
            make_objective(model_name, X_all_sc, y_all),
            n_trials=n_trials,
            show_progress_bar=False,
        )
        best_params = study.best_params
        model = fit_best_model(model_name, best_params, X_all_sc, y_all)

        final_models[target_key][model_name] = model
        final_params[target_key][model_name] = best_params


Final models: SF36_MCS: 100%|██████████| 6/6 [00:10<00:00,  1.81s/it]


In [7]:
joblib.dump(loocv_predictions,   RESULTS_DIR / 'loocv_predictions.pkl')
joblib.dump(final_models,        RESULTS_DIR / 'final_models.pkl')
joblib.dump(final_preprocessor,  RESULTS_DIR / 'final_preprocessor.pkl')
joblib.dump(final_params,        RESULTS_DIR / 'final_params.pkl')
joblib.dump(FEATURE_NAMES,       RESULTS_DIR / 'feature_names.pkl')


['results/feature_names.pkl']